# Model Complexity Comparison

**Research question:** Does increasing model complexity improve rental prediction performance?

This experiment compares a mean baseline, linear models, and tree ensembles with the same listing-only features (`make_listing`, no auxiliary tables) and fixed temporal split. The goal is to examine linear versus nonlinear model behavior, not only select the lowest RMSE. A tree model's gain may suggest that the linear form is insufficient, but does not identify the cause.

Run Jupyter from the repository root. Dependencies are listed in `requirements.txt`.

In [1]:
import sys
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').is_dir():
    raise RuntimeError('Start Jupyter from the repository root.')
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge

from src.evaluation import evaluate
from src.features import (
    encode_high_card,
    feature_groups,
    make_listing,
    onehot_encoding,
    ordinal_encoding,
    scale_numerical,
)
from src.validation import load_data

## Shared data and features

`load_data()` reads the original training file, applies shared cleaning, and returns the fixed January 2021–April 2024 training period and May 2024–March 2025 validation period. No test rows are used. Every model receives the same listing-only columns from `make_listing`; no auxiliary table is attached. Encoders are fitted on training rows only.

In [2]:
train_df, valid_df = load_data()
X_train = make_listing(train_df)
X_valid = make_listing(valid_df)
y_train = train_df['MONTHLY_RENT']
y_valid = valid_df['MONTHLY_RENT']

assert X_train.index.equals(y_train.index)
assert X_valid.index.equals(y_valid.index)
assert X_train.columns.equals(X_valid.columns)

print(f"Training:   {len(train_df):,} rows, {train_df['RENT_APPROVAL_DATE'].min()} to {train_df['RENT_APPROVAL_DATE'].max()}")
print(f"Validation: {len(valid_df):,} rows, {valid_df['RENT_APPROVAL_DATE'].min()} to {valid_df['RENT_APPROVAL_DATE'].max()}")
print('Features:', list(X_train.columns))

Training:   117,777 rows, 2021-01 to 2024-04
Validation: 31,582 rows, 2024-05 to 2025-03
Features: ['FLOOR_AREA_SQM', 'flat_age', 'month_index', 'TOWN', 'FLAT_TYPE', 'FLAT_MODEL', 'BLOCK', 'STREET']


## Model set and initial parameters

Tree models use moderate starting configurations and fixed random seeds. Training time measures model `fit()` only; encoding is done beforehand. These are starting points for comparison, not an exhaustive tuning study. Extra Trees and histogram gradient boosting add different tree-building strategies.

Encoding per family:

- Dummy / Linear / Ridge: scaled numeric, low-card one-hot, high-card ordinal.
- Tree models: unscaled numeric, low-card ordinal, high-card ordinal.

In [3]:
models = {
    'Dummy Regressor': DummyRegressor(strategy='mean'),
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(
        n_estimators=200, max_depth=18, min_samples_leaf=5,
        random_state=42, n_jobs=-1
    ),
    'Extra Trees': ExtraTreesRegressor(
        n_estimators=300, max_depth=18, min_samples_leaf=5,
        random_state=42, n_jobs=-1
    ),
    'HistGradientBoosting': HistGradientBoostingRegressor(
        max_iter=300, learning_rate=0.08, max_leaf_nodes=31,
        l2_regularization=1.0, random_state=42
    ),
    'LightGBM': LGBMRegressor(
        n_estimators=500, learning_rate=0.05, num_leaves=31,
        random_state=42, n_jobs=-1, verbosity=-1
    ),
    'CatBoost': CatBoostRegressor(
        iterations=500, learning_rate=0.05, depth=8,
        loss_function='RMSE', random_seed=42, thread_count=-1, verbose=False
    ),
}
model_family = {
    'Dummy Regressor': 'Constant baseline',
    'Linear Regression': 'Linear',
    'Ridge Regression': 'Regularized linear',
    'Random Forest': 'Tree ensemble',
    'Extra Trees': 'Randomized tree ensemble',
    'HistGradientBoosting': 'Histogram gradient boosting',
    'LightGBM': 'Gradient-boosted trees',
    'CatBoost': 'Gradient-boosted trees',
}

In [4]:
LINEAR_MODELS = ('Dummy Regressor', 'Linear Regression', 'Ridge Regression')


def encode(X_fit, X_eval, linear):
    if linear:
        X_fit, X_eval = scale_numerical(X_fit, X_eval)
        X_fit, X_eval = onehot_encoding(X_fit, X_eval)
    else:
        X_fit, X_eval = ordinal_encoding(X_fit, X_eval, columns=feature_groups(X_fit)[1])
    return encode_high_card(X_fit, X_eval, method='ordinal')


encoded = {linear: encode(X_train, X_valid, linear) for linear in (True, False)}

results = []
fitted_models = {}
for name, regressor in models.items():
    model_X_train, model_X_valid = encoded[name in LINEAR_MODELS]
    start = time.perf_counter()
    regressor.fit(model_X_train, y_train)
    fit_seconds = time.perf_counter() - start
    predictions = regressor.predict(model_X_valid)
    metrics = evaluate(y_valid, predictions)
    assert np.isfinite(predictions).all(), f'{name} produced non-finite predictions'
    assert all(np.isfinite(value) for value in metrics.values()), f'{name} produced non-finite metrics'
    results.append({
        'Model': name, 'Family': model_family[name], **metrics,
        'Training Time (s)': fit_seconds,
    })
    fitted_models[name] = regressor
    print(f"{name}: fit {fit_seconds:.2f}s | RMSE {metrics['RMSE']:.2f}")

results_df = pd.DataFrame(results).sort_values('RMSE').reset_index(drop=True)
results_df.insert(0, 'RMSE Rank', np.arange(1, len(results_df) + 1))

/home/dev/miniconda3/envs/kaggle/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Dummy Regressor: fit 0.00s | RMSE 841.49
Linear Regression: fit 0.17s | RMSE 615.65


Ridge Regression: fit 0.08s | RMSE 615.73


Random Forest: fit 15.17s | RMSE 490.03


Extra Trees: fit 8.93s | RMSE 489.71


HistGradientBoosting: fit 1.13s | RMSE 481.22


LightGBM: fit 1.10s | RMSE 480.60


CatBoost: fit 3.70s | RMSE 481.24


## Performance comparison and interpretation

RMSE and MAE use rental currency units; positive Mean Error indicates over-prediction. Compare the linear rows with tree models to assess whether the same listing features benefit from nonlinear splits or interactions. A performance gap alone does not establish which feature relationship explains it.

In [5]:
# Export a reusable performance table beside this notebook.
output_dir = PROJECT_ROOT / 'experiments'
csv_path = output_dir / 'model_comparison_results.csv'
markdown_path = output_dir / 'model_comparison_results.md'
tabm_results_path = output_dir / 'tabm_hdb_rent_results.csv'
results_df = results_df.drop(columns=['RMSE Rank'])
if tabm_results_path.exists():
    tabm_results = pd.read_csv(tabm_results_path)
    results_df = pd.concat([results_df, tabm_results], ignore_index=True)
results_df = results_df.sort_values('RMSE').reset_index(drop=True)
results_df.insert(0, 'RMSE Rank', np.arange(1, len(results_df) + 1))
display(results_df.round({
    'RMSE': 2, 'MAE': 2, 'R2': 4, 'Mean Error': 2, 'Training Time (s)': 2,
}))
results_df.to_csv(csv_path, index=False)
table = results_df.round({
    'RMSE': 2, 'MAE': 2, 'R2': 4, 'Mean Error': 2, 'Training Time (s)': 2,
})
headers = list(table.columns)
markdown_rows = ['| ' + ' | '.join(headers) + ' |', '| ' + ' | '.join(['---'] * len(headers)) + ' |']
markdown_rows += [
    '| ' + ' | '.join(str(value) for value in row) + ' |'
    for row in table.itertuples(index=False, name=None)
]
note = (
    '\n\nNote: TabM uses the same fixed train/validation dates but a larger, different feature set than the listing-only models. '
    'Its run time is approximate (about 155 seconds, inferred from the progress log). Mean Error was not included in the supplied TabM output.'
)
markdown_path.write_text(
    '# Model Comparison Results\n\n' + '\n'.join(markdown_rows) + note + '\n',
    encoding='utf-8',
)
print(f'Saved {csv_path.relative_to(PROJECT_ROOT)} and {markdown_path.relative_to(PROJECT_ROOT)}')

,RMSE Rank,Model,Family,RMSE,MAE,R2,Mean Error,Training Time (s)
0,1,LightGBM,Gradient-boosted trees,480.60,357.10,0.4677,-42.02,1.10
1,2,HistGradientBoosting,Histogram gradient boosting,481.22,357.64,0.4663,-42.52,1.13
2,3,CatBoost,Gradient-boosted trees,481.24,357.23,0.4662,-41.85,3.70
3,4,Extra Trees,Randomized tree ensemble,489.71,364.74,0.4473,-46.53,8.93
4,5,Random Forest,Tree ensemble,490.03,365.58,0.4466,-51.88,15.17
5,6,Linear Regression,Linear,615.65,459.14,0.1265,351.75,0.17
6,7,Ridge Regression,Regularized linear,615.73,459.25,0.1262,351.81,0.08
7,8,Dummy Regressor,Constant baseline,841.49,685.09,-0.6320,-523.65,0.00


Saved experiments/model_comparison_results.csv and experiments/model_comparison_results.md


## Reproducibility notes

All models use the same original `data/train.csv` through `load_data()`, the same split, and the same listing-only features. Validation is used only to report metrics. Keep this comparison's feature and split policy fixed when interpreting results.